# Tutorial 4: Agents in LangChain

In this tutorial, we'll explore Agents in LangChain, which are autonomous entities capable of using tools and making decisions to accomplish tasks.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

load_dotenv()

llm = ChatGroq(model_name='qwen/qwen3-32b', temperature=0.1)
print("Setup complete.")

## 1. Understanding Agent Architecture

Agents in LangChain combine an LLM with a set of tools to perform tasks. The LLM decides which tools to use and how to interpret their outputs.

**Note on API history:** the way you build an agent has moved twice. The original `initialize_agent()` / `AgentExecutor` classes were deprecated in LangChain 0.2 and now live in the separate `langchain-classic` package (only install that if you're maintaining old code — you'll never need it here, but you may see it in older tutorials online). They were replaced by `langgraph.prebuilt.create_react_agent`, which is itself now deprecated as of LangGraph 1.0. The current recommended API is `create_agent()` from the `langchain` package, used throughout this tutorial — it runs on LangGraph internally and adds a middleware system (covered in Tutorial 24) for customizing agent behavior.

## 2. Exploring Different Types of Agents

### Zero-shot React Agent

In [ ]:
search = DuckDuckGoSearchRun()

@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

@tool
def web_search(query: str) -> str:
    """Search the web for current information."""
    try:
        return search.run(query)[:500]
    except Exception as e:
        return f"Search unavailable: {e}"

# create_agent is the current top-level API for building agents (see note above)
tools = [get_word_length, web_search]
zero_shot_agent = create_agent(model=llm, tools=tools)

result = zero_shot_agent.invoke({
    "messages": [HumanMessage(content="What is the square root of the year Pythagoras was born? Search for the year first.")]
})
print(result["messages"][-1].content[:300])

In [ ]:
# Web search tool is already defined above as web_search
# Tool is a core abstraction — import it from langchain_core, not langchain_community:
from langchain_core.tools import Tool
web_search_tool = Tool(
    name='DuckDuckGoSearch',
    func=search.run,
    description='Search the web for current information'
)
print("Web search tool defined:", web_search_tool.name)

### Conversational Agent

In [ ]:
# Conversational agent with chat history — LangGraph maintains message history automatically
conversational_agent = create_agent(
    model=llm,
    tools=[web_search, get_word_length],
    system_prompt='You are a helpful conversational assistant. Answer questions and remember context.'
)

result = conversational_agent.invoke({
    "messages": [HumanMessage(content="Hello! I'd like to learn about the theory of relativity. Can you summarize it briefly?")]
})
print(result["messages"][-1].content[:300])

### STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION Agent

In [ ]:
# Plan-and-execute style agent — prompt instructs the model to plan before acting
plan_execute_agent = create_agent(
    model=llm,
    tools=[web_search, get_word_length],
    system_prompt=(
        'You are a careful problem-solver. Before acting:\n'
        '1. Break the problem into steps\n'
        '2. Execute each step using available tools\n'
        '3. Combine results into a final answer'
    )
)

result = plan_execute_agent.invoke({
    "messages": [HumanMessage(content="Find out who invented the telephone and calculate the number of years between that invention and the first moon landing in 1969.")]
})
print(result["messages"][-1].content[:300])

## 3. Creating Custom Tools for Agents

In [ ]:
@tool
def get_word_length_tool(word: str) -> int:
    """Returns the number of characters in a word."""
    return len(word)

custom_tools = [web_search, get_word_length_tool]
custom_agent = create_agent(
    model=llm,
    tools=custom_tools,
    system_prompt='You are a versatile assistant with access to web search and word length tools.'
)

result = custom_agent.invoke({
    "messages": [HumanMessage(content="How many letters are in the name of the inventor of the World Wide Web?")]
})
print(result["messages"][-1].content[:200])

## 4. Implementing a Multi-tool Agent for Task Solving

In [ ]:
# Multi-tool agent using all available tools
all_tools = [web_search, get_word_length]
multi_tool_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=(
        'You are a multi-capable assistant. Use available tools to:\n'
        '1. Find the current temperature in New York City\n'
        '2. Calculate the square root of that temperature\n'
        '3. Report both results clearly'
    )
)

result = multi_tool_agent.invoke({
    "messages": [HumanMessage(content="Find the current temperature in New York City and calculate its square root.")]
})
print(result["messages"][-1].content[:300])

## Conclusion

In this tutorial, we've explored Agents in LangChain, including different types of agents, creating custom tools, and implementing a multi-tool agent for complex task solving. Agents provide a powerful way to create autonomous systems that can leverage various tools and make decisions to accomplish tasks.